| **Field**              | **Information   |
|------------------------|-----------------|
| **Name**               |      Ayesha Ameer           |
| **Registration Number**|      04102213021           |
| **Project Title**      |       GenAI Domain Assistant - Vector Databases & Semantic Search          |
| **Week**               |        Week 11         |
| **Instructor**         |      Sir Faiz Ahmad           |

In [1]:
!pip install openai python-dotenv

# PART 1: EMBEDDINGS

## Task 1.1: Generate Embeddings

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# OpenRouter client setup
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

def get_embedding(text):
    """
    Generate embeddings using OpenRouter
    """

    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )

    return response.data[0].embedding


# Test
text = "vacation policy"

embedding = get_embedding(text)

print(f"Embedding length: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding length: 3072
First 5 values: [-0.031982421875, 0.016998291015625, -0.004550933837890625, -0.0034160614013671875, 0.03485107421875]


## Task 1.2: Calculate Similarity

In [2]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors

    Returns:
        Float between -1 and 1
    """

    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    dot_product = np.dot(vec1, vec2)

    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    # Avoid division by zero
    if norm1 == 0 or norm2 == 0:
        return 0.0

    return dot_product / (norm1 * norm2)


# Test phrases
phrases = [
    "vacation policy",
    "time off rules",
    "PTO guidelines",
    "dress code requirements"
]

# Generate embeddings
embeddings = [get_embedding(p) for p in phrases]

# Base phrase embedding
base = embeddings[0]

print(f'Comparing "{phrases[0]}" with:\n')

# Compare similarities
for i, phrase in enumerate(phrases[1:], start=1):

    similarity = cosine_similarity(base, embeddings[i])

    print(f"{phrase:30} Similarity: {similarity:.4f}")

Comparing "vacation policy" with:

time off rules                 Similarity: 0.6401
PTO guidelines                 Similarity: 0.3734
dress code requirements        Similarity: 0.2563


Expected: 'time off' and 'PTO' should have HIGH similarity (~0.85-0.95). 'dress code' should be LOW (~0.1-0.3). 

# PART 2: CHROMADB SETUP

## Task 2.1: Initialize ChromaDB

In [3]:
import os
import chromadb
import requests
from chromadb.api.types import EmbeddingFunction

# -----------------------------
# OpenRouter Embedding Wrapper
# -----------------------------
class OpenRouterEmbeddingFunction(EmbeddingFunction):
    def __init__(self, api_key, model="openai/text-embedding-3-small"):
        self.api_key = api_key
        self.model = model

    def __call__(self, input):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        response = requests.post(
            "https://openrouter.ai/api/v1/embeddings",
            headers=headers,
            json={
                "model": self.model,
                "input": input
            }
        )

        if response.status_code != 200:
            raise Exception(f"Embedding API error: {response.text}")

        data = response.json()
        return [item["embedding"] for item in data["data"]]


# -----------------------------
# Setup
# -----------------------------
api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found")

chroma_client = chromadb.PersistentClient(path="./chroma_db")

embedding_fn = OpenRouterEmbeddingFunction(
    api_key=api_key,
    model="openai/text-embedding-3-small"
)

collection = chroma_client.get_or_create_collection(
    name="company_docs",
    embedding_function=embedding_fn,
    metadata={"description": "Company policy documents"}
)

print("Collection Name:", collection.name)
print("Total Documents:", collection.count())

Collection Name: company_docs
Total Documents: 3


## Task 2.2: Load and Index Documents

In [8]:
pip install -U langchain langchain-community langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -----------------------------
# Load documents
# -----------------------------
loader = DirectoryLoader(
    "company_docs/",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")

# -----------------------------
# Split into chunks
# -----------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

# -----------------------------
# Check existing data
# -----------------------------
existing_count = collection.count()

if existing_count == 0:
    ids = []
    texts = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        ids.append(f"doc_{i}_{hash(chunk.page_content)}")  # safer unique ID
        texts.append(chunk.page_content)

        metadatas.append({
            "source": chunk.metadata.get("source", "unknown"),
            "chunk_index": i
        })

    # -----------------------------
    # Add to ChromaDB
    # -----------------------------
    collection.add(
        documents=texts,
        ids=ids,
        metadatas=metadatas
    )

    print(f"✅ Added {len(chunks)} chunks to ChromaDB")

else:
    print(f"⚠️ Collection already contains {existing_count} documents. Skipping ingestion.")

C:\Users\Ayesha\AppData\Local\Temp\ipykernel_16300\932578087.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Loaded 3 documents
Total chunks created: 4
⚠️ Collection already contains 3 documents. Skipping ingestion.


# PART 3: SEMANTIC RAG

## Task 3.1: Test Vector Search

In [5]:
def vector_search(query, n_results=3):
    """
    Semantic search using ChromaDB
    """
    return collection.query(
        query_texts=[query],
        n_results=n_results
    )


def print_search_results(query, results):
    """
    Pretty-print search results safely
    """
    print("\n" + "=" * 70)
    print(f"🔎 Query: {query}")
    print("=" * 70)

    documents = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0]

    if not documents:
        print("No results found.")
        return

    for i, doc in enumerate(documents):
        distance = distances[i] if i < len(distances) else None

        print(f"\nResult {i + 1}")
        if distance is not None:
            print(f"Similarity score (distance): {distance:.4f}")

        print("-" * 50)
        print(doc[:300] + "...")


# -----------------------------
# Test semantic understanding
# -----------------------------
test_queries = [
    "time off policy",   # should match vacation policy
    "WFH guidelines",    # should match remote work policy
    "maternity leave"    # should match parental leave
]

for query in test_queries:
    results = vector_search(query, n_results=2)
    print_search_results(query, results)


🔎 Query: time off policy

Result 1
Similarity score (distance): 0.9083
--------------------------------------------------
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Employees may work remotely up to 3 days per week.
Remote work requires manager approval.

Parental Le...

Result 2
Similarity score (distance): 1.2749
--------------------------------------------------
IT & Security Policy

Password Policy:
Employees must use strong passwords and change them every 90 days.

Device Usage:
Company devices should be used for work-related tasks only.

Data Security:
Do not share confidential company data externally without approval.

Software Installation:
Only approv...

🔎 Query: WFH guidelines

Result 1
Similarity score (distance): 1.5176
--------------------------------------------------
IT & Security Policy

Password Policy:
Employees m

Magic! 'time off' finds 'vacation', 'WFH' finds 'remote work', even though the exact words don't match!

## Task 3.2: Build Semantic RAG Pipeline

In [6]:
from langchain_openai import ChatOpenAI


# -----------------------------
# LLM Setup (OpenRouter compatible)
# -----------------------------
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)


# -----------------------------
# RAG Pipeline
# -----------------------------
def semantic_rag(query, n_results=3):
    """
    Complete semantic RAG pipeline with ChromaDB + OpenRouter
    """

    # Step 1: Vector search
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    documents = results.get("documents", [[]])[0]

    if not documents:
        return "❌ No relevant information found in the knowledge base."

    # Step 2: Build context (safe + limited)
    context = "\n\n---\n\n".join(documents[:n_results])

    # Step 3: System + user prompt (better RAG behavior)
    messages = [
        {
            "role": "system",
            "content": (
                "You are an HR assistant. "
                "Answer ONLY using the provided context. "
                "If the answer is not in the context, say: "
                "'I don't have enough information in the documents.'"
            )
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{query}

Answer clearly and concisely:
"""
        }
    ]

    # Step 4: Generate response
    response = llm.invoke(messages)

    return response.content


# -----------------------------
# Test Queries
# -----------------------------
test_questions = [
    "How much time off do employees get?",
    "Can I work from home?",
    "What is the maternity leave policy?"
]

for q in test_questions:
    print("\n" + "=" * 60)
    print(f"Q: {q}")
    print("=" * 60)
    print(semantic_rag(q))


Q: How much time off do employees get?
All full-time employees receive 15 days of paid vacation per year.

Q: Can I work from home?
Employees may work remotely up to 3 days per week with manager approval.

Q: What is the maternity leave policy?
12 weeks paid parental leave for primary caregivers.
6 weeks paid leave for secondary caregivers.


## BONUS: KEYWORD VS SEMANTIC COMPARISON

In [11]:
from collections import Counter
import re


# -----------------------------
# Improved Keyword Search
# -----------------------------
def keyword_search(query, chunks, top_k=3):
    """
    Simple TF-style keyword scoring (improved version)
    """
    query_words = re.findall(r'\w+', query.lower())
    scored = []

    for chunk in chunks:
        text = chunk.page_content.lower()
        text_words = re.findall(r'\w+', text)

        text_counter = Counter(text_words)

        # score = sum of query word frequencies in chunk
        score = sum(text_counter[word] for word in query_words)

        if score > 0:
            scored.append((score, chunk))

    scored.sort(key=lambda x: x[0], reverse=True)

    return [chunk for score, chunk in scored[:top_k]]


# -----------------------------
# Semantic search wrapper (safe)
# -----------------------------
def semantic_search(query, n_results=3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    docs = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0]

    return list(zip(docs, distances))


# -----------------------------
# Comparison function
# -----------------------------
def compare_search(query, chunks):
    print("\n" + "=" * 80)
    print(f"🔎 QUERY: {query}")
    print("=" * 80)

    # ---------------- Keyword ----------------
    print("\n🔤 KEYWORD SEARCH RESULTS:")
    kw_results = keyword_search(query, chunks, top_k=2)

    if not kw_results:
        print("No keyword matches found.")
    else:
        for i, chunk in enumerate(kw_results):
            print(f"\nResult {i+1}")
            print(chunk.page_content[:200] + "...")

    # ---------------- Semantic ----------------
    print("\n🧠 SEMANTIC SEARCH RESULTS:")
    sem_results = semantic_search(query, n_results=2)

    if not sem_results:
        print("No semantic matches found.")
    else:
        for i, (doc, dist) in enumerate(sem_results):
            print(f"\nResult {i+1}")
            print(f"Distance: {dist:.4f}")
            print(doc[:200] + "...")


# -----------------------------
# Run comparison
# -----------------------------
query = "PTO policy"  # should map to vacation policy

compare_search(query, chunks)


🔎 QUERY: PTO policy

🔤 KEYWORD SEARCH RESULTS:

Result 1
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Em...

Result 2
PTO (Paid Time Off) means vacation leave and sick leave combined.
Time off includes PTO, vacation days, and personal leave....

🧠 SEMANTIC SEARCH RESULTS:

Result 1
Distance: 1.3404
IT & Security Policy

Password Policy:
Employees must use strong passwords and change them every 90 days.

Device Usage:
Company devices should be used for work-related tasks only.

Data Security:
Do ...

Result 2
Distance: 1.3477
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Em...


Result: Keyword search finds 0 results (no 'PTO' in docs). Semantic search finds vacation policy (understands PTO = vacation)!

#  Learning Outcomes Report  
## Semantic Search and Retrieval-Augmented Generation (RAG) System using ChromaDB, LangChain, and OpenRouter



## 1. Introduction

In this activity, I developed a complete **Retrieval-Augmented Generation (RAG) system** using modern NLP tools including **ChromaDB (vector database)**, **LangChain (document processing framework)**, and **OpenRouter / OpenAI-compatible models**.  

The main objective was to understand how **semantic search differs from keyword-based search**, and how embeddings and large language models improve information retrieval systems.



## 2. Tools and Technologies Used

- Python  
- ChromaDB (Vector Database)  
- LangChain (Document loaders and text splitters)  
- OpenRouter API / OpenAI-compatible LLMs  
- Embedding Models (`text-embedding-3-large`)  
- Recursive Character Text Splitter  
- Vector Similarity Search  



## 3. Key Activities Performed

### 3.1 Document Loading and Preprocessing
- Loaded HR policy documents from a local directory.
- Used `DirectoryLoader` and `TextLoader` for ingestion.
- Structured raw documents for processing.



### 3.2 Text Chunking
- Applied `RecursiveCharacterTextSplitter` to split documents into smaller chunks.
- Used chunk size and overlap to preserve context.
- Learned how chunking impacts retrieval quality and embedding performance.



### 3.3 Vector Embedding and Storage
- Converted text chunks into vector embeddings using `text-embedding-3-large`.
- Stored embeddings in **ChromaDB persistent vector database**.
- Understood how vector databases represent semantic meaning instead of keywords.



### 3.4 Semantic Search Implementation
- Implemented similarity search using ChromaDB.
- Retrieved documents based on meaning rather than exact keyword matches.
- Observed semantic relationships such as:
  - “PTO” → “vacation leave”
  - “time off” → “vacation policy”



### 3.5 Keyword vs Semantic Search Comparison
- Built a custom keyword-based search using frequency scoring.
- Compared it with semantic vector search.
- Observed:
  - Keyword search fails on synonyms.
  - Semantic search captures meaning but does not always produce perfect equivalence.



### 3.6 RAG Pipeline Development
- Built a full Retrieval-Augmented Generation pipeline:
  - Vector retrieval (ChromaDB)
  - Context construction
  - LLM-based response generation (OpenRouter)
- Learned how LLMs generate answers grounded in retrieved context.



## 4. Key Learning Outcomes

### 4.1 Understanding Semantic Search
- Semantic search is based on **contextual meaning**, not exact words.
- Similar texts are represented as nearby vectors in embedding space.
- Similarity scores represent relative closeness, not probability.



### 4.2 Limitations of Keyword Search
- Keyword search relies on exact word matching.
- It fails when synonyms are used (e.g., PTO vs vacation).
- It can produce misleading results due to common words like “policy”.



### 4.3 Role of Embeddings
- Embeddings convert text into high-dimensional vectors.
- Similar meanings are placed closer in vector space.
- Learned that similarity scores are **not absolute values but relative distances**.



### 4.4 Importance of Data Quality
- Chunking strategy strongly affects retrieval performance.
- Lack of contextual explanation reduces embedding accuracy.
- Adding definitions and contextual sentences improves semantic alignment.



### 4.5 Understanding RAG Systems
- RAG combines:
  - Retrieval (ChromaDB)
  - Generation (LLMs)
- The model answers using retrieved context, reducing hallucinations.
- Proper context engineering is critical for accuracy.



### 4.6 Real-World Insights
- Production systems use:
  - Hybrid search (keyword + semantic)
  - Reranking models
  - Synonym expansion and ontology mapping
- Embeddings alone are not sufficient for domain-specific precision.



## 5. Challenges Faced

- LangChain version compatibility issues  
- Understanding embedding similarity interpretation  
- Unexpected behavior in keyword-based scoring  
- Difference between theoretical and real-world semantic similarity  



## 6. Conclusion

This activity provided hands-on experience in building a modern **AI-powered semantic search and RAG system**. It demonstrated how vector databases and embeddings enable meaning-based retrieval, and how LLMs enhance the final response generation.

The experiment highlighted that while semantic search is powerful, achieving high accuracy requires:
- High-quality data
- Proper chunking
- Hybrid retrieval strategies
- Careful prompt and context engineering



## 7. Future Improvements

- Implement hybrid search (BM25 + embeddings)
- Add reranking using cross-encoder or LLMs
- Improve synonym handling using domain ontology mapping
- Introduce evaluation metrics (Precision, Recall, MRR)
- Build a UI using Streamlit or FastAPI